In [ ]:
from google.colab import files
uploaded = files.upload()
uploaded.keys()

Saving audiofeatures_Ses1.csv to audiofeatures_Ses1.csv
Saving audiofeatures_Ses2.csv to audiofeatures_Ses2.csv
Saving audiofeatures_Ses3.csv to audiofeatures_Ses3.csv


dict_keys(['audiofeatures_Ses1.csv', 'audiofeatures_Ses2.csv', 'audiofeatures_Ses3.csv'])

In [ ]:
import pandas as pd

path = "audiofeatures_Ses1.csv"

df_try = pd.read_csv(path, engine="python")
print("Columns:", df_try.columns.tolist()[:20])
print("Shape:", df_try.shape)
df_try.head(2)

Columns: ['PATIENT_ID', 'F1', 'F2', 'F3', 'F1_BW', 'F2_BW', 'F3_BW', 'ANTIF1', 'ANTIF2', 'ANTIF3', 'ANTIF1_BW', 'ANTIF2_BW', 'ANTIF3_BW', 'FO', 'TO', 'HNR', 'CHNR', 'NNE', 'GNE', 'JITA']
Shape: (259, 38)


,PATIENT_ID,F1,F2,F3,F1_BW,F2_BW,F3_BW,ANTIF1,ANTIF2,ANTIF3,...,FFTR,FTRI,FATR,ATRI,label,AUDIO_MATERIAL,SESSION,audiopath,CPP,GROUP
0,5,757.473655,1125.995149,2805.771142,313.884026,181.793741,306.295249,980.732984,1916.709422,2829.387383,...,0.008998,4.668534,1.493603,10.17501,Sept,a2,ses1,data/data_final/Audios/Sept/A2/1/Sept_ses1_a2_...,[20.86710851 21.50437732 21.99647951 22.338656...,Sept
1,5,716.787736,1146.468364,2648.870247,303.267423,140.842098,242.502033,1004.062977,1922.644497,2704.648916,...,0.019709,10.175010,2.017783,10.17501,Sept,a3,ses1,data/data_final/Audios/Sept/A3/1/Sept_ses1_a3_...,[23.39600637 23.52497854 24.90003437 24.455056...,Sept


In [ ]:
import pandas as pd
import numpy as np

def load_csv(path):
    df = pd.read_csv(path, engine="python")
    df.columns = [c.strip() for c in df.columns]
    return df

ses1 = load_csv("audiofeatures_Ses1.csv")
ses2 = load_csv("audiofeatures_Ses2.csv")
ses3 = load_csv("audiofeatures_Ses3.csv")

print(ses1.shape, ses2.shape, ses3.shape)
print("ses1 columns:", ses1.columns.tolist())

(259, 38) (156, 38) (255, 38)
ses1 columns: ['PATIENT_ID', 'F1', 'F2', 'F3', 'F1_BW', 'F2_BW', 'F3_BW', 'ANTIF1', 'ANTIF2', 'ANTIF3', 'ANTIF1_BW', 'ANTIF2_BW', 'ANTIF3_BW', 'FO', 'TO', 'HNR', 'CHNR', 'NNE', 'GNE', 'JITA', 'JITTER', 'RAP', 'PPQ', 'SPPQ', 'SHDB', 'SHIMMER', 'APQ', 'SAPQ', 'FFTR', 'FTRI', 'FATR', 'ATRI', 'label', 'AUDIO_MATERIAL', 'SESSION', 'audiopath', 'CPP', 'GROUP']


In [ ]:
import pandas as pd
import numpy as np

df = pd.concat([ses1, ses2, ses3], ignore_index=True)

df.columns = [c.strip() for c in df.columns]
df["PATIENT_ID"] = pd.to_numeric(df["PATIENT_ID"], errors="coerce")
df = df[df["PATIENT_ID"].notna()].copy()
df["PATIENT_ID"] = df["PATIENT_ID"].astype(int)

print("df shape:", df.shape)
print("\nSESSION counts:\n", df["SESSION"].value_counts(dropna=False))
print("\nlabel counts:\n", df["label"].value_counts(dropna=False))
print("\nGROUP counts:\n", df["GROUP"].value_counts(dropna=False))
print("\nNulls in key cols:", df[["PATIENT_ID","SESSION","label","GROUP","CPP"]].isna().sum().to_dict())

df shape: (670, 38)

SESSION counts:
 SESSION
ses1    259
ses3    255
ses2    156
Name: count, dtype: int64

label counts:
 label
Tonsill    243
Contr      242
Sept       185
Name: count, dtype: int64

GROUP counts:
 GROUP
Tonsill    243
Contr      242
Sept       185
Name: count, dtype: int64

Nulls in key cols: {'PATIENT_ID': 0, 'SESSION': 0, 'label': 0, 'GROUP': 0, 'CPP': 0}


In [ ]:
import numpy as np
import pandas as pd
import ast

def parse_cpp(x):
    if isinstance(x, (list, np.ndarray)):
        arr = np.array(x, dtype=float)
        return arr

    s = str(x).strip()

    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)):
            return np.array(v, dtype=float)
    except Exception:
        pass

    nums = np.fromstring(s.replace("["," ").replace("]"," "), sep=" ")
    return nums.astype(float)

cpp_arr = df["CPP"].apply(parse_cpp)

lens = cpp_arr.apply(len)
print("CPP length stats:", lens.min(), lens.median(), lens.max())
print("Any zero-length CPP?", (lens==0).sum())

df["CPP_len"] = lens
df["CPP_mean"] = cpp_arr.apply(lambda a: float(np.mean(a)) if len(a) else np.nan)
df["CPP_std"]  = cpp_arr.apply(lambda a: float(np.std(a)) if len(a) else np.nan)
df["CPP_min"]  = cpp_arr.apply(lambda a: float(np.min(a)) if len(a) else np.nan)
df["CPP_max"]  = cpp_arr.apply(lambda a: float(np.max(a)) if len(a) else np.nan)
df["CPP_med"]  = cpp_arr.apply(lambda a: float(np.median(a)) if len(a) else np.nan)
df["CPP_p10"]  = cpp_arr.apply(lambda a: float(np.percentile(a,10)) if len(a) else np.nan)
df["CPP_p90"]  = cpp_arr.apply(lambda a: float(np.percentile(a,90)) if len(a) else np.nan)

print(df[["CPP_len","CPP_mean","CPP_std","CPP_min","CPP_max","CPP_med","CPP_p10","CPP_p90"]].describe())

CPP length stats: 3 149.0 519
Any zero-length CPP? 0
          CPP_len    CPP_mean     CPP_std     CPP_min     CPP_max     CPP_med  \
count  670.000000  670.000000  670.000000  670.000000  670.000000  670.000000   
mean   157.126866   22.497568    1.868783   17.371468   27.150349   22.530788   
std     51.403824    2.763591    0.481504    2.847917    2.993525    2.803509   
min      3.000000   15.811504    1.032081   11.116294   16.994565   15.978110   
25%    128.000000   20.486941    1.534510   15.319540   24.963785   20.487092   
50%    149.000000   22.208542    1.773498   17.226532   26.923707   22.238893   
75%    175.000000   24.198367    2.081321   19.090072   29.056715   24.275140   
max    519.000000   32.544489    5.678596   28.441331   36.224604   32.565337   

          CPP_p10     CPP_p90  
count  670.000000  670.000000  
mean    20.132674   24.850922  
std      2.729459    2.903883  
min     13.852589   16.882990  
25%     18.241807   22.713138  
50%     19.907350   24.56

/tmp/ipython-input-2188142063.py:22: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  nums = np.fromstring(s.replace("["," ").replace("]"," "), sep=" ")


In [ ]:
core_feats = [
    "CPP_mean", "CPP_std",
    "HNR", "JITTER", "SHIMMER", "FO"
]

pivot = df.pivot_table(
    index="PATIENT_ID",
    columns="SESSION",
    values=core_feats,
    aggfunc="mean"
)

required_sessions = ["ses1", "ses3"]
mask = pivot.columns.get_level_values(1).isin(required_sessions)
pivot = pivot.loc[:, mask]

pivot.columns = [f"{f}_{s}" for f, s in pivot.columns]

for f in core_feats:
    pivot[f"Δ_{f}"] = abs(pivot[f"{f}_ses3"] - pivot[f"{f}_ses1"])

print("Patients with ses1 & ses3:", pivot.shape[0])
pivot.head()

Patients with ses1 & ses3: 87


,CPP_mean_ses1,CPP_mean_ses3,CPP_std_ses1,CPP_std_ses3,FO_ses1,FO_ses3,HNR_ses1,HNR_ses3,JITTER_ses1,JITTER_ses3,SHIMMER_ses1,SHIMMER_ses3,Δ_CPP_mean,Δ_CPP_std,Δ_HNR,Δ_JITTER,Δ_SHIMMER,Δ_FO
PATIENT_ID,,,,,,,,,,,,,,,,,,
5,22.555946,23.392434,2.329214,1.962372,104.082881,105.804229,12.107307,10.495609,1.110303,0.792823,3.789627,5.556460,0.836488,0.366842,1.611698,0.317480,1.766833,1.721348
7,22.660793,21.624924,1.388098,1.417070,186.982355,173.958898,11.355498,13.630080,1.834280,0.948097,4.263528,3.013810,1.035870,0.028972,2.274581,0.886183,1.249718,13.023457
8,21.461058,25.421890,2.103461,3.219021,111.167561,105.668709,13.521368,8.512354,0.971247,1.174087,4.682943,3.461990,3.960832,1.115559,5.009013,0.202840,1.220953,5.498853
11,21.539963,22.170952,2.638378,1.306681,1028.497940,246.753194,0.188353,9.604326,18.728487,3.756831,24.761440,5.316333,0.630989,1.331697,9.415973,14.971657,19.445107,781.744746
12,26.727579,22.852444,2.116601,4.144325,165.157821,165.805047,14.099885,14.392125,0.682236,1.109482,3.202609,3.304441,3.875135,2.027724,0.292240,0.427245,0.101831,0.647226


In [ ]:
from sklearn.preprocessing import StandardScaler

delta_cols = [
    "Δ_CPP_mean",
    "Δ_CPP_std",
    "Δ_F0",
    "Δ_HNR",
    "Δ_JITTER",
    "Δ_SHIMMER"
]

delta_cols = [c for c in delta_cols if c in pivot.columns]

scaler = StandardScaler()
pivot_z = pivot.copy()
pivot_z[delta_cols] = scaler.fit_transform(pivot[delta_cols])

pivot_z[delta_cols].describe()

,Δ_CPP_mean,Δ_CPP_std,Δ_HNR,Δ_JITTER,Δ_SHIMMER
count,8.500000e+01,8.500000e+01,8.500000e+01,8.500000e+01,8.500000e+01
mean,-1.959217e-17,-8.751170e-17,-8.163405e-17,4.440892e-17,-1.900441e-16
std,1.005935e+00,1.005935e+00,1.005935e+00,1.005935e+00,1.005935e+00
min,-1.323869e+00,-8.892051e-01,-1.268175e+00,-5.463806e-01,-5.854340e-01
25%,-7.364277e-01,-6.018550e-01,-6.755839e-01,-4.071517e-01,-4.241383e-01
50%,-1.394764e-01,-3.366542e-01,-1.702240e-01,-2.794868e-01,-2.382789e-01
75%,5.000672e-01,2.420635e-01,5.528446e-01,2.381573e-02,3.961232e-02
max,2.872731e+00,5.649882e+00,4.840899e+00,7.950410e+00,7.017757e+00


In [ ]:
pivot_z["CHANGE_SCORE"] = pivot_z[delta_cols].sum(axis=1)

pivot_z["CHANGE_SCORE"].describe()

,CHANGE_SCORE
count,8.700000e+01
mean,-2.067312e-16
std,3.210170e+00
min,-3.199018e+00
25%,-1.854005e+00
50%,-6.634609e-01
75%,8.189313e-01
max,2.236804e+01


In [ ]:
extreme = pivot_z.sort_values("CHANGE_SCORE", ascending=False)

extreme.head(10)[
    ["CHANGE_SCORE", "Δ_CPP_mean", "Δ_CPP_std", "Δ_HNR", "Δ_JITTER", "Δ_SHIMMER"]
]

,CHANGE_SCORE,Δ_CPP_mean,Δ_CPP_std,Δ_HNR,Δ_JITTER,Δ_SHIMMER
PATIENT_ID,,,,,,
11,22.368042,-0.838712,3.397688,4.840899,7.950410,7.017757
63,6.382869,0.824456,1.037604,1.434313,-0.350751,3.437246
8,5.867828,1.772642,2.698313,1.945083,-0.435735,-0.112475
12,5.342360,1.705436,5.649882,-1.154313,-0.308311,-0.550333
80,4.891720,0.639724,-0.386212,0.586008,1.405170,2.647030
23,3.785616,-0.228577,-0.080784,0.324983,1.674778,2.095216
33,3.778970,2.516424,1.465185,-0.390418,0.560472,-0.372693
103,3.734550,2.812374,-0.806818,1.557169,0.245082,-0.073257
50,3.534215,0.500067,-0.084272,2.572022,0.440008,0.106390


In [ ]:
selected_ids = [11, 63, 8, 12, 80, 23]

df_extreme = df[
    (df["PATIENT_ID"].isin(selected_ids)) &
    (df["SESSION"].isin(["ses1", "ses3"]))
].sort_values(["PATIENT_ID", "SESSION"])

df_extreme[["PATIENT_ID", "SESSION", "label", "audiopath"]]

,PATIENT_ID,SESSION,label,audiopath
3,8,ses1,Sept,data/data_final/Audios/Sept/A1/1/Sept_ses1_a1_...
4,8,ses1,Sept,data/data_final/Audios/Sept/A3/1/Sept_ses1_a3_...
5,8,ses1,Sept,data/data_final/Audios/Sept/A2/1/Sept_ses1_a2_...
418,8,ses3,Sept,data/data_final/Audios/Sept/A3/3/Sept_ses3_a3_...
419,8,ses3,Sept,data/data_final/Audios/Sept/A1/3/Sept_ses3_a1_...
420,8,ses3,Sept,data/data_final/Audios/Sept/A2/3/Sept_ses3_a2_...
178,11,ses1,Tonsill,data/data_final/Audios/Tonsill/A1/1/Tonsill_se...
179,11,ses1,Tonsill,data/data_final/Audios/Tonsill/A3/1/Tonsill_se...
180,11,ses1,Tonsill,data/data_final/Audios/Tonsill/A2/1/Tonsill_se...
592,11,ses3,Tonsill,data/data_final/Audios/Tonsill/A3/3/Tonsill_se...


In [ ]:
extreme_patients = extreme[extreme["GROUP_ses1"].isin(["Sept","Tonsill"])] if "GROUP_ses1" in extreme.columns else extreme
extreme_patients.head(10)

,CPP_mean_ses1,CPP_mean_ses3,CPP_std_ses1,CPP_std_ses3,FO_ses1,FO_ses3,HNR_ses1,HNR_ses3,JITTER_ses1,JITTER_ses3,SHIMMER_ses1,SHIMMER_ses3,Δ_CPP_mean,Δ_CPP_std,Δ_HNR,Δ_JITTER,Δ_SHIMMER,Δ_FO,CHANGE_SCORE
PATIENT_ID,,,,,,,,,,,,,,,,,,,
11,21.539963,22.170952,2.638378,1.306681,1028.497940,246.753194,0.188353,9.604326,18.728487,3.756831,24.761440,5.316333,-0.838712,3.397688,4.840899,7.950410,7.017757,781.744746,22.368042
63,21.282815,24.034577,1.990225,2.592552,87.700865,100.699083,8.804537,4.572832,1.976738,2.329243,2.540161,12.833843,0.824456,1.037604,1.434313,-0.350751,3.437246,12.998218,6.382869
8,21.461058,25.421890,2.103461,3.219021,111.167561,105.668709,13.521368,8.512354,0.971247,1.174087,4.682943,3.461990,1.772642,2.698313,1.945083,-0.435735,-0.112475,5.498853,5.867828
12,26.727579,22.852444,2.116601,4.144325,165.157821,165.805047,14.099885,14.392125,0.682236,1.109482,3.202609,3.304441,1.705436,5.649882,-1.154313,-0.308311,-0.550333,0.647226,5.342360
80,20.217975,17.701773,1.523388,1.685693,183.235894,176.469260,12.678689,9.737965,2.644087,6.088939,2.581166,10.855137,0.639724,-0.386212,0.586008,1.405170,2.647030,6.766634,4.891720
23,21.394077,22.803072,2.700821,2.957516,202.268745,152.025523,14.176592,11.633105,4.936936,1.017277,12.554966,5.691377,-0.228577,-0.080784,0.324983,1.674778,2.095216,50.243223,3.785616
33,21.660980,16.751720,2.458844,1.724376,96.212518,86.568500,9.447389,10.902152,0.856021,2.813278,4.795540,5.351401,2.516424,1.465185,-0.390418,0.560472,-0.372693,9.644018,3.778970
103,22.218052,27.504689,1.662976,1.630657,162.863246,154.979380,12.024933,7.606260,1.424133,2.825958,4.178639,2.857450,2.812374,-0.806818,1.557169,0.245082,-0.073257,7.883866,3.734550
50,20.666458,23.004578,1.863110,2.118727,101.144718,98.805889,8.089094,2.125981,2.222756,3.967865,4.499651,6.280000,0.500067,-0.084272,2.572022,0.440008,0.106390,2.338828,3.534215


In [ ]:
meta = (
    df[["PATIENT_ID", "SESSION", "GROUP", "label"]]
    .drop_duplicates()
)

meta_p = meta.pivot(index="PATIENT_ID", columns="SESSION", values=["GROUP", "label"])

meta_p.columns = [f"{a}_{b}" for a, b in meta_p.columns]

pivot_z = pivot_z.merge(meta_p, left_index=True, right_index=True, how="left")

pivot_z["group_changed"] = pivot_z.get("GROUP_ses1") != pivot_z.get("GROUP_ses3")
pivot_z["label_changed"] = pivot_z.get("label_ses1") != pivot_z.get("label_ses3")

print("group_changed count:", pivot_z["group_changed"].sum())
print("label_changed count:", pivot_z["label_changed"].sum())

extreme2 = pivot_z.sort_values("CHANGE_SCORE", ascending=False)

cols_to_show = [
    "CHANGE_SCORE",
    "GROUP_ses1", "label_ses1",
    "GROUP_ses3", "label_ses3",
    "A_CPP_mean","A_CPP_std","A_HNR","A_JITTER","A_SHIMMER"
]
cols_to_show = [c for c in cols_to_show if c in extreme2.columns]

extreme2[cols_to_show].head(15)

group_changed count: 2
label_changed count: 2


,CHANGE_SCORE,GROUP_ses1,label_ses1,GROUP_ses3,label_ses3
PATIENT_ID,,,,,
11,22.368042,Tonsill,Tonsill,Tonsill,Tonsill
63,6.382869,Tonsill,Tonsill,Tonsill,Tonsill
8,5.867828,Sept,Sept,Sept,Sept
12,5.342360,Contr,Contr,Contr,Contr
80,4.891720,Tonsill,Tonsill,Tonsill,Tonsill
23,3.785616,Sept,Sept,Sept,Sept
33,3.778970,Sept,Sept,Sept,Sept
103,3.734550,Contr,Contr,Contr,Contr
50,3.534215,Sept,Sept,Sept,Sept


In [ ]:
print("All labels:", sorted(df["label"].astype(str).str.strip().unique()))
print("\nLabels per session:")
print(df.assign(label_clean=df["label"].astype(str).str.strip())
        .groupby("SESSION")["label_clean"]
        .unique())

print("\nOverall label counts:")
print(df["label"].astype(str).str.strip().value_counts())

print("\nLabel counts by session:")
print(df.assign(label_clean=df["label"].astype(str).str.strip())
        .groupby(["SESSION","label_clean"])
        .size()
        .unstack(fill_value=0))

pivot_ids = pivot_z.index
df_pivot = df[df["PATIENT_ID"].isin(pivot_ids)].copy()
df_pivot["label_clean"] = df_pivot["label"].astype(str).str.strip()

print("\nLabels in pivot cohort (ses1&ses3 patients):")
print(df_pivot["label_clean"].value_counts())

print("\nLabels in pivot cohort by session:")
print(df_pivot.groupby(["SESSION","label_clean"]).size().unstack(fill_value=0))

mask_fess = df["label"].astype(str).str.lower().str.contains("fess", na=False)
print("\nRows with 'fess' in label (any session):", mask_fess.sum())
display(df.loc[mask_fess, ["PATIENT_ID","SESSION","label","GROUP"]].head(30))

All labels: ['Contr', 'Sept', 'Tonsill']

Labels per session:
SESSION
ses1    [Sept, Contr, Tonsill]
ses2          [Contr, Tonsill]
ses3    [Sept, Contr, Tonsill]
Name: label_clean, dtype: object

Overall label counts:
label
Tonsill    243
Contr      242
Sept       185
Name: count, dtype: int64

Label counts by session:
label_clean  Contr  Sept  Tonsill
SESSION                          
ses1            83    92       84
ses2            78     0       78
ses3            81    93       81

Labels in pivot cohort (ses1&ses3 patients):
label_clean
Tonsill    243
Contr      242
Sept       185
Name: count, dtype: int64

Labels in pivot cohort by session:
label_clean  Contr  Sept  Tonsill
SESSION                          
ses1            83    92       84
ses2            78     0       78
ses3            81    93       81

Rows with 'fess' in label (any session): 0


,PATIENT_ID,SESSION,label,GROUP
